In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

def train_and_save_model(data_path):
    # 1. Load and Filter
    df = pd.read_csv(data_path)
    relevant_types = ['TRANSFER', 'CASH_OUT']
    df = df[df['type'].isin(relevant_types)].copy()

    # 2. Feature Engineering (The PM 'Secret Sauce')
    # errorBalanceOrg captures when a transaction empties an account incorrectly
    df['errorBalanceOrg'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
    df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

    # 3. Encoding and Prep
    df_final = df.drop(['nameOrig', 'nameDest', 'isFlaggedFraud'], axis=1)
    df_final = pd.get_dummies(df_final, columns=['type'], drop_first=True)

    X = df_final.drop(['isFraud'], axis=1)
    y = df_final['isFraud']

    # 4. Train with High Sensitivity (Class Weights)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
    
    # We weight Fraud 100x more than Legit to ensure the model is 'aggressive'
    model = RandomForestClassifier(
        n_estimators=100, 
        max_depth=15, 
        class_weight={0: 1, 1: 100}, 
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)

    # 5. Export for the Frontend
    joblib.dump(model, 'fraud_model.pkl')
    joblib.dump(X.columns.tolist(), 'model_columns.pkl')
    return "Model and Columns saved successfully!"

if __name__ == "__main__":
    train_and_save_model('PS_20174392719_1491204439457_log.csv')

In [5]:
pip install pandas scikit-learn joblib

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ------------------ --------------------- 4.5/9.9 MB 30.4 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.9 MB 27.4 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 26.2 MB/s  0:00:00
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 8.1/8.1 MB 50.5 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------  12.3/12.4 MB 65.2 MB/s eta 0:00:01
   ---------------------------------------- 12.4/12.4 MB 56.4 MB/s  0:00:00
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   -------- -

In [21]:
!pip install streamlit

In [22]:
import joblib
# Save the model to a file
joblib.dump(model, 'fraud_model.pkl')

['fraud_model.pkl']